In [ ]:
import pandas as pd
import numpy as np

### Ensure num records == num files

In [ ]:
import os
base_directory = "/Users/madhu/Desktop/WFP/wfp_reports"
reports = pd.read_csv(base_directory+'/reports_metadata.csv')
counter = 0

def count_files_in_folders(base_path, counter):
    for root, dirs, files in os.walk(base_path):
        # Count only files (not subdirectories)
        num_files = len([f for f in files if os.path.isfile(os.path.join(root, f))])
        counter += num_files
    return counter
        # print(f"{root}: {num_files} file(s)")

# Example usage

counter = count_files_in_folders(base_directory, counter)
reports.shape[0] == counter

In [ ]:
reports.shape[0]

### Certain Reports seem to be repeated across multiple Categories

In [ ]:
reports['Topic'] = reports['Topic'].str.replace(r'[^A-Za-z\s]', '', regex=True).str.strip()

In [ ]:
repeated_reports = reports.groupby(['Report Name'])['Topic'].agg(unique_topics = lambda x : x.unique(),
                                                num_unique_topics = lambda x : x.nunique()).reset_index()

In [ ]:
repeated_reports = repeated_reports[repeated_reports['num_unique_topics'] > 1].sort_values(['num_unique_topics'],
                                                                                           ascending = False)

In [ ]:
for i in repeated_reports['unique_topics'][0:1]:
    print(i)

In [ ]:
repeated_reports = repeated_reports.assign(
    topics_exploded = repeated_reports["unique_topics"]
).explode("topics_exploded")


### Topic Pairs appearing together

In [ ]:
from itertools import combinations
import pandas as pd

# Step 2: get all topic pairs per report
pairs = []
for topics in repeated_reports['unique_topics']:
    if len(topics) > 1:
        pairs.extend(list(combinations(sorted(topics), 2)))

# Step 3: count frequency of each topic pair
pair_counts = pd.Series(pairs).value_counts().reset_index()
pair_counts.columns = ['Topic_Pair', 'Count']

pair_counts.head(10)


In [ ]:
reports['Topic'].nunique()

### PDF Data Extraction

In [ ]:
# import pymupdf  # PyMuPDF

# def extract_pdf_structure(pdf_path):
#     doc = pymupdf.open(pdf_path)
#     pages_output = []

#     for page in doc:
#         blocks = page.get_text("dict")["blocks"]
#         page_data = []

#         for b in blocks:
#             if "lines" not in b:
#                 continue
#             text = ""
#             for line in b["lines"]:
#                 for span in line["spans"]:
#                     text += span["text"] + " "
#                     font_size = span["size"]
#             bbox = b["bbox"]

#             page_data.append({
#                 "text": text.strip(),
#                 "bbox": bbox,
#                 "font_size": font_size,
#                 "page_num": page.number
#             })

#         pages_output.append(page_data)
#     return pages_output

# pdf_data = extract_pdf_structure("/Users/madhu/Desktop/WFP/wfp_reports/wfp_reports/Academia_and_think_tanks/2025_Anticipatory_Action_Learning_and_validation_R.pdf")
# pdf_data[:2]  # inspect sample
